# 2 — Freeze 21 variants, create 480-row manifest, import 24 controls

Do not run Stage 1 outcomes before this notebook completes. The complete Stage 0 bundle must be at `~/stage0`. This notebook exports the user-space native-library prefix before starting the isolated OOD Python process.

In [ ]:
import csv
import json
import os
import subprocess
from pathlib import Path

R = Path.home() / "async-vla-latency-bench"
P = Path.home() / "LIBERO-plus"
PY = Path.home() / "venv-stage1-ood/bin/python"
NATIVE = Path.home() / "stage1-native"
S0 = Path.home() / "stage0"
OUT = Path.home() / "stage1"
OUT.mkdir(exist_ok=True)

required = [
    S0 / "selected_high_delay.json",
    S0 / "latency_calibration_episode_results.csv",
    P / "libero/libero/assets",
    NATIVE / "lib",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise SystemExit("STOP: missing prerequisites: " + str(missing))
if json.loads((S0 / "selected_high_delay.json").read_text())["high_added_delay_ms"] != 200:
    raise SystemExit("STOP: frozen d* is not 200 ms")

env = os.environ.copy()
env.update({
    "PYTHONPATH": str(P),
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "MPLBACKEND": "Agg",
    "MAGICK_HOME": str(NATIVE),
    "PATH": str(NATIVE / "bin") + os.pathsep + env.get("PATH", ""),
    "LD_LIBRARY_PATH": str(NATIVE / "lib") + os.pathsep + env.get("LD_LIBRARY_PATH", ""),
    # The parent has already supplied the native prefix at process startup.
    "STAGE1_NATIVE_REEXEC": str(NATIVE),
})
print("OOD Python:", PY)
print("LD_LIBRARY_PATH:", env["LD_LIBRARY_PATH"])
subprocess.run(
    [str(PY), "-c", "from wand.api import library; print('preflight: MagickWand OK')"],
    env=env, check=True,
)


In [ ]:
VAR = OUT / "stage1_resolved_variants.csv"
MAN = OUT / "stage1_manifest.csv"
bench = subprocess.run(
    ["git", "-C", str(R), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
plus = subprocess.run(
    ["git", "-C", str(P), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.resolve_stage1_variants",
    "--output", str(VAR),
], cwd=R, env=env, check=True)
subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.make_stage1_manifest",
    "--variants", str(VAR),
    "--selected-delay", str(S0 / "selected_high_delay.json"),
    "--output", str(MAN),
    "--git-sha", bench,
    "--lerobot-git-sha", "2aba372b4e217cc47db28e0f836859b20d1456c9",
    "--libero-plus-git-sha", plus,
    "--model-revision", "8e174154ef5f6c60a8da12ae99c303d8963138c1",
], cwd=R, env=env, check=True)
subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.import_stage0_controls",
    "--manifest", str(MAN),
    "--stage0-dir", str(S0),
    "--stage1-dir", str(OUT),
], cwd=R, env=env, check=True)


In [ ]:
from collections import Counter

rows = list(csv.DictReader(open(OUT / "stage1_manifest.csv")))
results = list(csv.DictReader(open(OUT / "stage1_episode_results.csv")))
assert len(rows) == 480 and len({row['run_id'] for row in rows}) == 480
assert Counter(row['scene_condition'] for row in rows) == {'ood': 420, 'id': 60}
assert sum(row['reuse_stage0'].lower() == 'true' for row in rows) == 24
assert len(results) == 24
assert {row['seed'] for row in rows} == {'0', '1', '2', '3', '4'}
print("PASS manifest=480 OOD=420 ID=60 imported=24 new=456")
print("STOP HERE and paste this output plus the 21-row variant CSV for review before notebook 3.")
